# RAG with Weaviate

Time to connect everything together! You've been focusing on Weaviate for the retrieval (or "R") part of RAG. Weaviate can actually be used for the full RAG workflow&mdash;from beginning to end!

**Run the two cells below to install `weaviate-client` and create a client.**

In [ ]:
# Install weaviate-client...

import weaviate
from openai import OpenAI as OpenAIClient

raw_client = OpenAIClient()
API_KEY  = str(raw_client.api_key)
API_BASE = str(raw_client.base_url)

HEADERS = {
    "X-OpenAI-Api-Key": API_KEY,
    "X-OpenAI-BaseURL": API_BASE.rstrip("/").removesuffix("/v1"),
}

try:
    client = weaviate.connect_to_embedded(
        version="1.32.3",
        headers=HEADERS,
        environment_variables={"LOG_LEVEL": "error"},
    )
except weaviate.exceptions.WeaviateStartUpError:
    # If already running, connect to it instead
    client = weaviate.connect_to_local(
        port=8079,
        grpc_port=50050,
        headers=HEADERS,
    )

from weaviate.classes.config import Property, DataType, Configure, Tokenization

# Delete "Chunks" if it's already present to allow repeated runs
if "Chunks" in client.collections.list_all():
    client.collections.delete("Chunks")

client.collections.create(
    name="Chunks",
    properties=[
        Property(
            name="document_title",
            data_type=DataType.TEXT,
        ),
        Property(
            name="chunk",
            data_type=DataType.TEXT,
        ),
        Property(
            name="chunk_number",
            data_type=DataType.INT,
        ),
        Property(
            name="filename",
            data_type=DataType.TEXT,
            tokenization=Tokenization.FIELD
        ),
    ],
    vector_config=[
        Configure.Vectors.text2vec_openai(
            name="default",
            source_properties=["document_title", "chunk"],
            model="text-embedding-3-small"
        )
    ],
    generative_config=Configure.Generative.openai(
        model="gpt-4o-mini"
    )
)

## TASK 1
def get_chunks_by_length_with_overlap(src_text: str, chunk_length: int = 500, overlap: int = 100) -> list[str]:
    """
    Split text into chunks of approximately `chunk_length` characters.
    """
    chunks = []
    for i in range(0, len(src_text), chunk_length):
        chunks.append(src_text[i:i + chunk_length + overlap])
    return chunks

from pathlib import Path

filepath = Path("data/parsed/hai_ai-index-report-2025_chapter2_final-parsed-text.md")
raw_text = filepath.read_text(encoding="utf-8")
chunk_texts = get_chunks_by_length_with_overlap(raw_text)
print(f"Number of chunks: {len(chunk_texts)}")

chunks = client.collections.get("Chunks")

## TASK 2
from tqdm import tqdm
from weaviate.util import generate_uuid5

with chunks.batch.fixed_size(batch_size=100) as batch:
    for i, chunk in tqdm(enumerate(chunk_texts)):
        obj = {
            "document_title": "HAI AI Index Report 2025 - Chapter 2",
            "filename": filepath.name,
            "chunk": chunk,
            "chunk_number": i + 1,
        }

        # Add object to batch for import
        batch.add_object(
            properties=obj,
            uuid=generate_uuid5(obj),
        )

# Check for any failed objects during import
if chunks.batch.failed_objects:
    print(f"{len(chunks.batch.failed_objects)} objects failed during import:")
    for failed in chunks.batch.failed_objects[:3]:
        print(failed.message)
        
print(f"Collection size: {len(chunks)}")

## TASK 3
import textwrap

query = "latest developments in retrieval augmented generation (RAG)"

print(f"===== {query} =====")
response = chunks.query.hybrid(
    query=query,
    limit=5
)

for i, o in enumerate(response.objects):
    print(textwrap.indent(f"> Result {i + 1}:", "  "))
    wrapped_text = textwrap.fill(o.properties['chunk'][:200]+"...", width=80)
    print(textwrap.indent(wrapped_text, "    "))
    print()
    
import textwrap

query = "latest developments in retrieval augmented generation (RAG)"
task = "what developments have there been in RAG recently?"

response = chunks.generate.hybrid(
    query=query,
    limit=5,
    grouped_task=task
)

print(f"RAG response: \n{response.generative.text}.")

print(f"\nSupporting passages:")
for i, o in enumerate(response.objects):
    print(textwrap.indent(f"> Result {i + 1}:", "  "))
    wrapped_text = textwrap.fill(o.properties['chunk'][:200]+"...", width=80)
    print(textwrap.indent(wrapped_text, "    "))
    print()

## TASK 4
def task_to_query(task: str) -> str:
    from openai import OpenAI
    client = OpenAI()
    """
    Convert a task to a search query for Weaviate.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": f"""
                    Provide the answer only and nothing else at all.
                    To perform the below task, what search query should I use on
                    a vector database like Weaviate to find relevant information?

                    Task: {task}
                """,
            }
        ],
    )
    return response.choices[0].message.content.strip()

task = "what developments have there been in RAG recently?"
query = task_to_query(task)
print(f"Task to query: \nTask '{task}': \nQuery: '{query}'")

response = chunks.generate.near_text(
    query=query,  # Generated query from the RAG task
    limit=3,
    grouped_task=task  # RAG task - user input
)

print(f"RAG response: \n{response.generative.text}.")

client.close()


In [1]:
!pip install weaviate-client==4.16.10 pydantic==2.11.9

In [2]:
import weaviate
from openai import OpenAI as OpenAIClient

raw_client = OpenAIClient()
API_KEY  = str(raw_client.api_key)
API_BASE = str(raw_client.base_url)

HEADERS = {
    "X-OpenAI-Api-Key": API_KEY,
    "X-OpenAI-BaseURL": API_BASE.rstrip("/").removesuffix("/v1"),
}

try:
    client = weaviate.connect_to_embedded(
        version="1.32.3",
        headers=HEADERS,
        environment_variables={"LOG_LEVEL": "error"},
    )
except weaviate.exceptions.WeaviateStartUpError:
    # If already running, connect to it instead
    client = weaviate.connect_to_local(
        port=8079,
        grpc_port=50050,
        headers=HEADERS,
    )

### Data ingestion

We'll use pre-extracted data from the 2025 AI report. Just like we learned, let's create a collection for chunks of data.

**Configure the collection to use the `text-embedding-3-small` model for embeddings and the `gpt-4o-mini` generative model.**

In [4]:
from weaviate.classes.config import Property, DataType, Configure, Tokenization

# Delete "Chunks" if it's already present to allow repeated runs
if "Chunks" in client.collections.list_all():
    client.collections.delete("Chunks")

client.collections.create(
    name="Chunks",
    properties=[
        Property(
            name="document_title",
            data_type=DataType.TEXT,
        ),
        Property(
            name="chunk",
            data_type=DataType.TEXT,
        ),
        Property(
            name="chunk_number",
            data_type=DataType.INT,
        ),
        Property(
            name="filename",
            data_type=DataType.TEXT,
            tokenization=Tokenization.FIELD
        ),
    ],
    vector_config=[
        Configure.Vectors.text2vec_openai(
            name="default",
            source_properties=["document_title", "chunk"],
            model="____"
        )
    ],
    generative_config=Configure.____.____(
        model="____"
    )
)

### Document Processing

We'll chunk a large document and store chunks with vector embeddings, enabling semantic search and generation.

For this data, we'll chunk the data using a length-based method

**Run the code provided to chunk the data parsed PDF using a character length and overlap.**

In [5]:
def get_chunks_by_length_with_overlap(src_text: str, chunk_length: int = 500, overlap: int = 100) -> list[str]:
    """
    Split text into chunks of approximately `chunk_length` characters.
    """
    chunks = []
    for i in range(0, len(src_text), chunk_length):
        chunks.append(src_text[i:i + chunk_length + overlap])
    return chunks

from pathlib import Path

filepath = Path("data/parsed/hai_ai-index-report-2025_chapter2_excerpts-parsed-text.md")
raw_text = filepath.read_text(encoding="utf-8")
chunk_texts = get_chunks_by_length_with_overlap(raw_text)

print(f"Number of chunks: {len(chunk_texts)}")
print(f"First chunk:\n\n{chunk_texts[0]}")

**Run the following cell to batch insert the chunks into the collection.**

In [10]:
from tqdm import tqdm
from weaviate.util import generate_uuid5

chunks = client.collections.use("Chunks")

with chunks.batch.fixed_size(batch_size=100) as batch:
    for i, chunk in tqdm(enumerate(chunk_texts)):
        obj = {
            "document_title": "HAI AI Index Report 2025 - Chapter 2",
            "filename": filepath.name,
            "chunk": chunk,
            "chunk_number": i + 1,
        }

        # Add object to batch for import
        batch.add_object(
            properties=obj,
            uuid=generate_uuid5(obj),
        )

# Check for any failed objects during import
if chunks.batch.failed_objects:
    print(f"{len(chunks.batch.failed_objects)} objects failed during import:")
    for failed in chunks.batch.failed_objects[:3]:
        print(failed.message)

print(f"Collection size: {len(chunks)}")

### Retrieval

First, let's test how hybrid search finds relevant chunks for a query.

**Perform a hybrid search using the `query` variable provided.**

In [13]:
import textwrap

query = "latest developments in retrieval augmented generation (RAG)"

print(f"===== {query} =====")
response = ____(
    query=____,
    limit=5
)

for i, o in enumerate(response.objects):
    print(textwrap.indent(f"> Result {i + 1}:", "  "))
    wrapped_text = textwrap.fill(o.properties['chunk'][:200]+"...", width=80)
    print(textwrap.indent(wrapped_text, "    "))
    print()

### Full RAG pipeline

Now we'll use Weaviate's generative capabilities to create responses based on retrieved chunks. This combines retrieval with generation for a complete RAG system.

![images/llm_3_rag_weaviate.png](images/llm_3_rag_weaviate.png)

**Repeat the hybrid search, but this time, specifying a `task` requesting each RAG development from the document.**

In [14]:
import textwrap

query = "latest developments in retrieval augmented generation (RAG)"
task = "what developments have there been in RAG recently?"

response = chunks.generate.hybrid(
    query=query,
    limit=5,
    grouped_task=____
)

print(f"RAG response: \n{response.generative.text}.")

The retrieved data used for RAG is also available in the response:

In [15]:
print(f"\nSupporting passages:")
for i, o in enumerate(response.objects):
    print(textwrap.indent(f"> Result {i + 1}:", "  "))
    wrapped_text = textwrap.fill(o.properties['chunk'][:200]+"...", width=80)
    print(textwrap.indent(wrapped_text, "    "))
    print()

### Query generation

In a user-facing application like a chatbot, it may be too much to ask a user for the task (what they want) and the query (how to search it). 

For cases like this, you can use "query generation", where we use an LLM to generate a query from a user-specified task.

**Define a `task_to_query()` function to generate a query from a task, and use it to perform a vector search using the `task` variable provided.**

In [16]:
def ____(task: str) -> str:
    from openai import OpenAI
    client = OpenAI()
    """
    Convert a task to a search query for Weaviate.
    """
    response = client.chat.completions.____(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": f"""
                    Provide the answer only and nothing else at all.
                    To perform the below task, what search query should I use on
                    a vector database like Weaviate to find relevant information?

                    Task: {____}
                """,
            }
        ],
    )
    return response.choices[0].message.content.strip()

task = "what developments have there been in RAG recently?"
query = task_to_query(task)
print(f"Task to query: \nTask '{task}': \nQuery: '{query}'")

And this method can be applied like so to generate the final response.

In [17]:
task = "what developments have there been in RAG recently?"

response = chunks.generate.____(
    query=task_to_query(____),  # Generated query from the RAG task
    limit=3,
    grouped_task=task  # RAG task - user input
)

print(f"RAG response: \n{response.generative.text}.")

In [18]:
client.close()